# T2 — Storytelling Dashboard (Subplot Grid) · `HARD`

**Task:** Choose a dataset with at least 3 numeric and 2 categorical columns.  
Build a **2×2 subplot figure** that tells a cohesive story — each panel must connect to a central theme.  
Apply a consistent color palette, shared legends, proper annotations, and **export at 300 DPI**.  
Write a **3-sentence narrative summary** at the top of the notebook explaining the story.

---

### Story Theme: *"Who Survived the Titanic — and Why?"*

**Narrative Summary**

The Titanic dataset reveals that survival was not random — it was deeply stratified by class, gender, and wealth.  
Women and children were evacuated first, giving them dramatically higher survival odds regardless of ticket class.  
Taken together, the four panels below paint a consistent portrait: your odds of survival were largely determined before the ship even set sail.


In [ ]:
# ── Imports & global dark theme ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

PALETTE = ["#2a9d8f", "#e76f51", "#f4a261", "#264653", "#e9c46a", "#a8dadc"]
BG, TEXT, GRID = "#0d1117", "#e6edf3", "#21262d"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT,
    "axes.titlecolor": TEXT, "xtick.color": TEXT, "ytick.color": TEXT,
    "text.color": TEXT, "grid.color": GRID, "grid.linestyle": "--",
    "grid.alpha": 0.5, "legend.facecolor": "#161b22",
    "legend.edgecolor": GRID, "font.family": "sans-serif", "font.size": 11,
})
sns.set_theme(style="dark", rc=plt.rcParams)

df = sns.load_dataset("titanic")
print(f"Dataset loaded: {df.shape}")


In [ ]:
# ── 2×2 Dashboard ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10), facecolor=BG)
fig.suptitle("Who Survived the Titanic — and Why?",
             fontsize=18, fontweight="bold", color=TEXT, y=0.98)

# gridspec gives precise control over spacing between panels
gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

for ax in [ax1, ax2, ax3, ax4]:
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)

# ── Panel 1: Survival rate by sex ─────────────────────────────────────────────
# Color maps to gender stereotype intentionally subverted by data (men died more)
surv_sex = df.groupby("sex")["survived"].mean() * 100
bars = ax1.bar(surv_sex.index, surv_sex.values,
               color=[PALETTE[0], PALETTE[3]],  # female=teal, male=dark
               edgecolor=BG, width=0.5, zorder=3)
for bar, val in zip(bars, surv_sex.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f"{val:.1f}%", ha="center", fontweight="bold")
ax1.set_ylim(0, 100)
ax1.set_ylabel("Survival Rate (%)")
ax1.set_title("① Survival Rate by Sex", fontweight="bold")
ax1.yaxis.grid(True, zorder=0)
# Arrow annotation pointing to female bar — highlights the key finding
ax1.annotate("Women evacuated first
→ 74% survived",
             xy=(0, surv_sex["female"]), xytext=(0.6, 60),
             textcoords="data", fontsize=9, color=PALETTE[4],
             arrowprops=dict(arrowstyle="->", color=PALETTE[4], lw=1.2))

# ── Panel 2: Survival rate by passenger class ─────────────────────────────────
surv_class = df.groupby("pclass")["survived"].mean() * 100
bars2 = ax2.bar([1,2,3], surv_class.values,
                color=[PALETTE[0], PALETTE[2], PALETTE[1]],
                edgecolor=BG, width=0.5, zorder=3)
for bar, val in zip(bars2, surv_class.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f"{val:.1f}%", ha="center", fontweight="bold")
ax2.set_xticks([1,2,3])
ax2.set_xticklabels(["1st Class","2nd Class","3rd Class"])
ax2.set_ylim(0, 75)
ax2.set_ylabel("Survival Rate (%)")
ax2.set_title("② Survival Rate by Passenger Class", fontweight="bold")
ax2.yaxis.grid(True, zorder=0)

# ── Panel 3: Age KDE split by survival ────────────────────────────────────────
# KDE lets us compare age distributions as smooth curves — cleaner than histograms here
for survived, label, color in [(0, "Did not survive", PALETTE[1]),
                                (1, "Survived",        PALETTE[0])]:
    ages = df[df["survived"] == survived]["age"].dropna()
    sns.kdeplot(ages, ax=ax3, color=color, linewidth=2.5,
                label=label, fill=True, alpha=0.20)
ax3.set_xlabel("Age (years)")
ax3.set_ylabel("Density")
ax3.set_title("③ Age Distribution by Survival Outcome", fontweight="bold")
ax3.legend(facecolor="#161b22", edgecolor=GRID, labelcolor=TEXT)

# ── Panel 4: Heatmap — survival rate by sex × class ───────────────────────────
# Cross-tabulation exposes interaction effects between two categorical variables
pivot = df.pivot_table(values="survived", index="sex",
                       columns="pclass", aggfunc="mean") * 100
sns.heatmap(pivot, annot=True, fmt=".1f", ax=ax4,
            cmap=sns.light_palette(PALETTE[0], as_cmap=True),
            linewidths=0.4, linecolor=BG,
            annot_kws={"size": 13, "weight": "bold", "color": BG},
            cbar_kws={"label": "Survival Rate (%)"})
ax4.set_xlabel("Passenger Class")
ax4.set_ylabel("Sex")
ax4.set_title("④ Survival Rate: Sex × Class", fontweight="bold")
ax4.set_xticklabels(["1st","2nd","3rd"])

# Export at 300 DPI as required by the task
plt.savefig("t2_dashboard_300dpi.png", dpi=300, bbox_inches="tight", facecolor=BG)
plt.show()
print("✅ Dashboard saved at 300 DPI → t2_dashboard_300dpi.png")
